In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [0]:
train_df = pd.read_csv("train_trip_duration_features.csv")
test_df = pd.read_csv("test_trip_duration_features.csv")

TARGET = 'trip_duration_minutes'

X_train = train_df.drop(columns = [TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns = [TARGET])
y_test = test_df[TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")



In [0]:
categorical_cols = ['priority_level', 'status','payment_type','booking_source']

numeric_cols = ['fare_amount','distance','expected_vs_actual_pickup_minutes_difference','dispatch_to_arrival_minutes','month','quarter', 'year', 'weekend_flag',  'pickup_latitude', 'pickup_longitude', 'fare_per_distance', 'pickup_delay_signal', 'journey_urgency', 'lat_delta', 'lon_delta', 'location_distance_proxy']



In [0]:
# sklearn preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
)

# Model pipeline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())   
])

In [0]:
# MLflow experiement and train inside tracked run
mlflow.set_experiment("/team1-trip-duration-prediction")

# Handle NaN values (journey_urgency has missing values)
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

with mlflow.start_run(run_name="linear_regression_baseline") as run:
    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("num_features", len(numeric_cols) + len(categorical_cols))
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("target", TARGET)

    # Train
    lr_pipeline.fit(X_train, y_train)

    # Predict
    y_pred = lr_pipeline.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Log metrics
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    # Log the full pipeline (preprocessor + model)
    mlflow.sklearn.log_model(lr_pipeline, "model")

    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²:   {r2:.3f}")

In [0]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='hotpink', label='Perfect')
plt.xlabel("Actual Trip Duration (min)")
plt.ylabel("Predicted Trip Duration (min)")
plt.title("Linear Regression: Predicted vs Actual")
plt.legend()
plt.show()

In [0]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest pipeline 
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=20))
])

with mlflow.start_run(run_name="random_forest") as rf_run:
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("num_features", len(numeric_cols) + len(categorical_cols))
    mlflow.log_param("target", TARGET)

    rf_pipeline.fit(X_train, y_train)
    rf_pred = rf_pipeline.predict(X_test)

    rf_mae = mean_absolute_error(y_test, rf_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
    rf_r2 = r2_score(y_test, rf_pred)

    mlflow.log_metric("mae", rf_mae)
    mlflow.log_metric("rmse", rf_rmse)
    mlflow.log_metric("r2", rf_r2)

    mlflow.sklearn.log_model(rf_pipeline, "model")

    rf_run_id = rf_run.info.run_id
    print(f"Run ID: {rf_run_id}")
    print(f"MAE:  {rf_mae:.3f}")
    print(f"RMSE: {rf_rmse:.3f}")
    print(f"R²:   {rf_r2:.3f}")

In [0]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae, rf_mae],
    "RMSE": [rmse, rf_rmse],
    "R²": [r2, rf_r2]
})

display(comparison)

In [0]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_pred, alpha=0.3, s=10, color='teal')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='hotpink', label='Perfect')
plt.xlabel("Actual Trip Duration (min)")
plt.ylabel("Predicted Trip Duration (min)")
plt.title("Random Forest: Predicted vs Actual")
plt.legend()
plt.show()

In [0]:
# Cap extreme outliers (trips > 120 min are likely data errors)
OUTLIER_CAP = 120

train_mask = y_train <= OUTLIER_CAP
test_mask = y_test <= OUTLIER_CAP

X_train_capped = X_train[train_mask]
y_train_capped = y_train[train_mask]
X_test_capped = X_test[test_mask]
y_test_capped = y_test[test_mask]

print(f"Removed {(~train_mask).sum()} train / {(~test_mask).sum()} test outliers (>{OUTLIER_CAP} min)")
print(f"Remaining: Train {X_train_capped.shape[0]}, Test {X_test_capped.shape[0]}")

# Log-transform the target to handle skew
y_train_log = np.log1p(y_train_capped)
y_test_log = np.log1p(y_test_capped)

# Train Random Forest on log-transformed target
rf_log_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=20))
])

with mlflow.start_run(run_name="random_forest_log_target") as rf_log_run:
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("target_transform", "log1p")
    mlflow.log_param("outlier_cap_minutes", OUTLIER_CAP)
    mlflow.log_param("num_features", len(numeric_cols) + len(categorical_cols))
    mlflow.log_param("target", TARGET)

    rf_log_pipeline.fit(X_train_capped, y_train_log)

    # Predict in log space, then convert back
    y_pred_log = rf_log_pipeline.predict(X_test_capped)
    y_pred_actual = np.expm1(y_pred_log)

    # Metrics on original scale
    log_mae = mean_absolute_error(y_test_capped, y_pred_actual)
    log_rmse = np.sqrt(mean_squared_error(y_test_capped, y_pred_actual))
    log_r2 = r2_score(y_test_capped, y_pred_actual)

    mlflow.log_metric("mae", log_mae)
    mlflow.log_metric("rmse", log_rmse)
    mlflow.log_metric("r2", log_r2)
    mlflow.sklearn.log_model(rf_log_pipeline, "model")

    print(f"\nRun ID: {rf_log_run.info.run_id}")
    print(f"MAE:  {log_mae:.3f}")
    print(f"RMSE: {log_rmse:.3f}")
    print(f"R²:   {log_r2:.3f}")

# Compare all three
print("\n=== Model Comparison ===")
print(f"{'Model':<35} {'MAE':>6} {'RMSE':>7} {'R²':>6}")
print(f"{'Linear Regression':<35} {mae:>6.3f} {rmse:>7.3f} {r2:>6.3f}")
print(f"{'Random Forest':<35} {rf_mae:>6.3f} {rf_rmse:>7.3f} {rf_r2:>6.3f}")
print(f"{'RF + Log-Transform + Cap':<35} {log_mae:>6.3f} {log_rmse:>7.3f} {log_r2:>6.3f}")

In [0]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test_capped, y_pred_actual, alpha=0.3, s=10, color='darkorange')
plt.plot([y_test_capped.min(), y_test_capped.max()], [y_test_capped.min(), y_test_capped.max()], '--', color='hotpink', label='Perfect')
plt.xlabel("Actual Trip Duration (min)")
plt.ylabel("Predicted Trip Duration (min)")
plt.title("RF + Log-Transform + Outlier Cap: Predicted vs Actual")
plt.legend()
plt.show()

In [0]:
from mlflow.models import infer_signature

# Set to True only when you intentionally want to register a new model version
REGISTER_MODEL = False

model_name = "students_data.team1_taxi.trip_duration_model"

if REGISTER_MODEL:
    # Re-log model with signature (required by Unity Catalog)
    signature = infer_signature(X_test, y_pred)

    with mlflow.start_run(run_id=run_id):
        mlflow.sklearn.log_model(lr_pipeline, "model", signature=signature)

    model_uri = f"runs:/{run_id}/model"
    result = mlflow.register_model(model_uri, model_name)
    print(f"Model registered: {model_name}, version {result.version}")
else:
    print("Skipping model registration (set REGISTER_MODEL = True to register a new version)")